# 🎬 CrossSync - Sincronizador Semántico de Subtítulos

Este notebook te permite sincronizar subtítulos desincronizados en castellano usando un archivo de referencia en inglés (u otro idioma).
Utiliza un modelo de embeddings multilingüe (`SentenceTransformers`) para alinear semánticamente las oraciones y opcionalmente un LLM (local vía `Ollama` o remoto vía `Gemini`) para realizar cortes precisos de diálogos.

### 🚀 Instrucciones:
1. **Entorno de ejecución**: Ve a *Entorno de ejecución* > *Cambiar tipo de entorno de ejecución* y selecciona **T4 GPU** para que Ollama y la generación de embeddings funcionen a máxima velocidad.
2. Ejecuta la celda de **Instalación** para preparar el entorno.
3. Sube tus archivos `.srt` usando el panel de archivos de Colab (menú izquierdo) o ejecútalos directamente si usas rutas locales.
4. Configura las opciones en el formulario de la celda de **Ejecución** y ejecuta para sincronizar.

In [ ]:
#@title 🛠️ Paso 1: Instalación de Dependencias y Preparación

import os

# Clonar repositorio si no está presente
repositorio_git = "https://github.com/HichamLL04/CrossSync.git" #@param {type:"string"}
nombre_carpeta = "CrossSync"

if not os.path.exists(nombre_carpeta):
    print(f"Clonando el repositorio {repositorio_git}...")
    !git clone {repositorio_git}
    %cd {nombre_carpeta}
else:
    %cd {nombre_carpeta}
    !git pull

# Instalar dependencias necesarias
print("Instalando dependencias...")
!pip install -q pysrt sentence-transformers numpy requests torch

In [ ]:
#@title 🦙 Paso 2: Iniciar Ollama en Segundo Plano (Opcional)
#@markdown Si deseas usar un LLM local en Colab para los cortes semánticos, marca esta opción e indica el modelo.
usar_ollama = True #@param {type:"boolean"}
modelo_ollama = "qwen2.5:3b" #@param {type:"string"}

if usar_ollama:
    print("Instalando Ollama...")
    !curl -fsSL https://ollama.com/install.sh | sh
    
    print("Iniciando el servidor de Ollama en segundo plano...")
    import subprocess
    import time
    subprocess.Popen(["ollama", "serve"])
    time.sleep(5)  # Esperar a que inicie el servidor
    
    print(f"Descargando el modelo {modelo_ollama}...")
    !ollama pull {modelo_ollama}
    print("Ollama listo y configurado!")
else:
    print("Ollama omitido. Se usará la alineación semántica por embeddings y cortes proporcionales si es necesario.")

In [ ]:
#@title ⚙️ Paso 3: Configurar y Ejecutar la Sincronización
#@markdown Introduce las rutas de tus archivos subidos a Colab. Puedes arrastrar tus archivos al panel izquierdo de Colab para subirlos y copiar su ruta.

subtitulo_mal_sincronizado = "/content/mal_sincronizado.srt" #@param {type:"string"}
subtitulo_referencia = "/content/referencia.srt" #@param {type:"string"}
subtitulo_resultado = "/content/resultado_sincronizado.srt" #@param {type:"string"}

proveedor_llm = "gemini" #@param ["ollama", "gemini", "none"]
modelo_llm = "gemini-2.5-flash" #@param {type:"string"}
api_key = "" #@param {type:"string"}
servidor_ollama = "http://localhost:11434" #@param {type:"string"}

# Verificar que los archivos de entrada existen antes de proceder
if not os.path.exists(subtitulo_mal_sincronizado):
    print(f"⚠️ Error: No se encuentra el archivo mal sincronizado en: {subtitulo_mal_sincronizado}")
    print("Por favor, sube tu archivo a Colab y actualiza la ruta.")
elif not os.path.exists(subtitulo_referencia):
    print(f"⚠️ Error: No se encuentra el archivo de referencia en: {subtitulo_referencia}")
    print("Por favor, sube tu archivo a Colab y actualiza la ruta.")
else:
    # Construir comando de sincronización
    cmd = f"PYTHONPATH=. python sync.py --unsynced \"{subtitulo_mal_sincronizado}\" --synced \"{subtitulo_referencia}\" --output \"{subtitulo_resultado}\""
    
    if proveedor_llm == "ollama" and usar_ollama:
        cmd += f" --llm-provider ollama --llm-model {modelo_llm} --ollama-url {servidor_ollama}"
    elif proveedor_llm == "gemini" and api_key:
        cmd += f" --llm-provider gemini --llm-model {modelo_llm} --api-key {api_key}"
        
    print(f"Ejecutando comando: {cmd}")
    !{cmd}
    
    if os.path.exists(subtitulo_resultado):
        print(f"🎉 ¡Éxito! El archivo sincronizado se ha guardado en: {subtitulo_resultado}")
        
        # Botón para descargar el archivo directamente en tu PC
        from google.colab import files
        files.download(subtitulo_resultado)
    else:
        print("⚠️ Ocurrió un error y el archivo de salida no pudo ser generado.")